# Running Stage 1: Stream & Group — Jupyter Notebook


## Prerequisites
An existing 'uv virtual environment' should be installed with the the dependencies required to run the the various scripts in the codebase. This can be done by following the instructions in the "install_guide.md" file (located in the demo directory, the same directory as this script).

## Dev Cell (for advanced users)
Run a quick GPU driver check, this is necessary for the ML Training and Inference section(s) but will need to be run early on to ensure environment compatibility

In [ ]:
!cd ..
import sys
print("Python:", sys.executable)

import torch
print("PyTorch:", torch.__version__)


if hasattr(torch, "accelerator") and torch.accelerator.is_available():
    _accel = torch.accelerator.current_accelerator()
    print(f"Accelerator: {_accel.type} (devices: {torch.accelerator.device_count()})")
elif torch.cuda.is_available():
    print(f"CUDA available (devices: {torch.cuda.device_count()})")
else:
    print("No GPU accelerator detected — will use CPU")

Consult **Troubleshooting** if path doesn't mention `.venv`

## Select data paths
Define the config.yaml path, the name of the intermediate json and the name of the output stats json. If unsure, do not change the defaults.


In [ ]:
CONFIG_PATH = "pre_process/config.yaml"
OUT_DATA    = "intermediate_grouper.json"
OUT_STATS   = "pipeline_stats.json"

### (Note for developers): 
For this early release of the codebase the archived files of the plankton (.tar) files will need to be downloaded locally from the blob. In the next release, functionality should be added to allow direct azure blob storage access. The temporary caching has been hardcoded for this release due to time constraints which is why Out_data needs to be defined

## Load Settings
Double check the config.yaml file and ensure that you are happy with all the path directories and settings.
The following Cell of code will read these settings and prep the pipeline for the codebase.

In [ ]:
from pre_process._pre_process_utils.pipeline_utils import load_config
from pre_process.tar_streamer import StreamConfig, ConcurrencyConfig
from pre_process.resolution_grouper import GrouperConfig

data = load_config(CONFIG_PATH)

stream_cfg  = StreamConfig.from_dict(data)
conc_cfg    = ConcurrencyConfig.from_dict(data)
grouper_cfg = GrouperConfig.from_dict(data)

print(f"Archives to process: {len(stream_cfg.tar_paths)}")
print(f"Tile size:           {grouper_cfg.tile_size}")

## Streaming images from Archives
Image streaming is setup in the next code cell, this will allow the images to be read efficiently by considering your PC's hardware capabilities (this is done automatically but note that this is one of the first core sections performance will differ from machine to machine)

In [ ]:
import time
from pre_process.tar_streamer import TarImageStream

t1 = time.perf_counter()
streamer = TarImageStream(config=stream_cfg, concurrency=conc_cfg)
print(f"Streamer ready in {time.perf_counter() - t1:.1f}s")
print(f"Stream stats: {streamer.stats}")

## Group image by image resolution
Images are grouped together by their dimensions (width x height), a core prerequisite required by the zarr and ome-zarr sections in the latter stages of the pre-processing section of the pipeline.

In [ ]:
from pre_process.resolution_grouper import ResolutionGrouper

t2 = time.perf_counter()
grouper = ResolutionGrouper(grouper_cfg).ingest(streamer)
dt = time.perf_counter() - t2

print(f"Grouping complete in {dt:.1f}s")
print(f"Total images: {grouper.total_images}")
print(f"Buckets:      {len(grouper)}")
print()
for entry in grouper.summary():
    print(f"  {entry['bucket']}: {entry['count']} images")

## Export results
Two files are generated, the first are the grouped images themselves and the second being a log detailing the the parameters and success of this stage, this is to allow for problems to be isolated quicker in the future, leading to faster fixes to the codebase as a whole.

In [ ]:
from pre_process._pre_process_utils.pipeline_utils import safe_write_json

stats = {
    "timing": {"stream_and_group_s": round(dt, 2)},
    "stream_stats": streamer.stats,
    "grouper_summary": grouper.summary(),
}
grouper_data = {str(k): v for k, v in dict(grouper).items()}

safe_write_json(grouper_data, OUT_DATA)
safe_write_json(stats, OUT_STATS)

print(f"Data  saved to: {OUT_DATA}")
print(f"Stats saved to: {OUT_STATS}")

## Stats Sanity check (Opt)
Run the following script to print the saved stats and check if the generated results deviate exponentially from the expected results

In [ ]:
import json
from pathlib import Path

stats_check = json.loads(Path(OUT_STATS).read_text())
print(json.dumps(stats_check, indent=2))

## Troubleshooting

### Problem: .venv not shown in the file path indicates that Jupyter cannot find the libraries that were installed for the project


!uv add --dev ipykernel
!uv run ipython kernel install --user \
  --env VIRTUAL_ENV "$(pwd)/.venv" \
  --name planktoshare \
  --display-name "Planktoshare (.venv)"